# Problem Book P1, Chapter 1: Two Tomes at a Time (Python companion)

Part A checks every number of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import itertools, math, random
from fractions import Fraction as Fr
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
random.seed(1)

## Part A: checks

**Problem 1.1 (Sicherman's dice).** We search *all* pairs of dice with faces in $\{1,\dots,11\}$ (a face larger than $11$
is impossible, since the other die shows at least $1$) whose sum has the law of two ordinary dice, and factor
$(x+\dots+x^6)^2$.

In [ ]:
from collections import Counter
target = Counter(a + b for a in range(1, 7) for b in range(1, 7))
dice = list(itertools.combinations_with_replacement(range(1, 12), 6))
found = set()
for A in dice:
    if sum(A) > 42 - 6:          # the two dice together have face sum 42
        continue
    rest = 42 - sum(A)
    for B in dice:
        if sum(B) == rest and B >= A and Counter(a + b for a in A for b in B) == target:
            found.add((A, B))
print(found)
x = sp.symbols('x')
print(sp.factor(sum(x**k for k in range(1, 7))**2))

**Problem 1.2 (loaded dice).** The polynomial $1+x+\dots+x^{10}$ has no real root; and a random search confirms that the
AM--GM bound is never beaten.

In [ ]:
print(np.roots([1] * 11))       # all non-real
best = 1
for _ in range(20000):
    p = np.random.dirichlet(np.ones(6)); q = np.random.dirichlet(np.ones(6))
    s = np.convolve(p, q)
    best = min(best, np.max(np.abs(s - 1 / 11)))
print("closest any random pair gets to uniform (sup distance):", best)

**Problem 1.3 (Schur).** The primes dividing some value of $X^2+1$ are $2$ and the primes $\equiv1\pmod4$; a quick look.

In [ ]:
divs = set()
for n in range(2000):
    divs |= set(sp.primefactors(n * n + 1))
small = sorted(p for p in divs if p < 120)
print(small)
print(all(p == 2 or p % 4 == 1 for p in divs))

**Problem 1.4 (functions on $\mathbb F_p$).** Interpolate every permutation of $\mathbb F_5$ and check the degree bound, and
the bound for its powers.

In [ ]:
p = 5
X = sp.symbols('X')
def interp(f):
    P = sum(f[a] * (1 - (X - a)**(p - 1)) for a in range(p))
    return sp.Poly(sp.expand(P), X, modulus=p)
degs = Counter()
ok = True
for perm in itertools.permutations(range(p)):
    degs[interp(perm).degree()] += 1
    for t in range(1, p - 1):
        ok &= interp([pow(v, t, p) for v in perm]).degree() <= p - 2
print(degs, ok)
print(interp([pow(a, 3, p) for a in range(p)]))

**Problem 1.5 (Fibonacci modulo $p$).** Pisano periods for all $p<2000$, against the divisibility of Problem 1.5(b).

In [ ]:
def pisano(p):
    a, b, n = 0, 1, 0
    while True:
        a, b = b, (a + b) % p
        n += 1
        if (a, b) == (0, 1):
            return n
bad = []
for p in sp.primerange(3, 2000):
    if p == 5:
        continue
    per = pisano(p)
    m = p - 1 if sp.legendre_symbol(5, p) == 1 else 2 * (p + 1)
    if m % per or sp.fibonacci(p - sp.legendre_symbol(5, p)) % p:
        bad.append(p)
print("exceptions:", bad, "| periods for 7 and 11:", pisano(7), pisano(11))

**Problem 1.6 (which square root).** Partial sums of the binomial series modulo $7^{12}$ and $2^{20}$.

In [ ]:
def series_mod(x, m, N=80):
    s = Fr(0)
    for n in range(N):
        s += Fr(sp.binomial(sp.Rational(1, 2), n)) * x**n
    return s.numerator * pow(s.denominator, -1, m) % m
m7, m2 = 7**12, 2**20
s7, s2 = series_mod(7, m7), series_mod(8, m2)
print("Q_7:", s7 % 7, (s7 * s7 - 8) % m7)
print("Q_2:", (s2 - (-3)) % m2, (s2 * s2 - 9) % m2)
print([sp.binomial(sp.Rational(1, 2), n) * 2**(2 * n - 1) * (-1)**(n - 1) == sp.catalan(n - 1) for n in range(1, 10)])

**Problem 1.7 (Pascal modulo 2).** Odd entries per row, and in the first $2^k$ rows.

In [ ]:
rows = [[1]]
for n in range(255):
    r = rows[-1]
    rows.append([1] + [(r[i] + r[i + 1]) % 2 for i in range(len(r) - 1)] + [1])
print(all(sum(rows[n]) == 2**bin(n).count('1') for n in range(256)))
print([sum(sum(rows[n]) for n in range(2**k)) for k in range(9)], [3**k for k in range(9)])

**Problems 1.8 and 1.9 (Millin, Sylvester).**

In [ ]:
import mpmath as mp
mp.mp.dps = 40
print(mp.nsum(lambda n: 1 / mp.fib(2**int(n)), [0, 9]), (7 - mp.sqrt(5)) / 2)
s = [2]
for _ in range(7):
    s.append(s[-1]**2 - s[-1] + 1)
print(sum(Fr(1, v) for v in s[:6]) == 1 - Fr(1, s[6] - 1), all(math.gcd(a, b) == 1 for a, b in itertools.combinations(s, 2)))
u = mp.mpf(s[7]) - mp.mpf(1) / 2
E = mp.e**(mp.log(u) / 2**8)
print(E, [int(mp.floor(E**(2**(k + 1)) + 0.5)) for k in range(7)] == s[:7])
def greedy(fr):
    out = []
    while fr:
        n = -(-fr.denominator // fr.numerator)
        out.append(n); fr -= Fr(1, n)
    return out
print(greedy(Fr(4, 13)))

**Problems 1.10 to 1.12 (arithmetic functions).**

In [ ]:
for n in range(1, 11):
    G = sp.Matrix(n, n, lambda i, j: math.gcd(i + 1, j + 1))
    G2 = sp.Matrix(n, n, lambda i, j: math.gcd(i + 1, j + 1)**2)
    J2 = lambda m: m * m * sp.prod([1 - sp.Rational(1, q * q) for q in sp.primefactors(m)])
    assert G.det() == sp.prod([sp.totient(k) for k in range(1, n + 1)])
    assert G2.det() == sp.prod([J2(k) for k in range(1, n + 1)])
print("Smith: ok;  det(gcd^2) for n=4:", sp.Matrix(4, 4, lambda i, j: math.gcd(i + 1, j + 1)**2).det())
print(sum(Fr(int(sp.totient(n)), 2**n - 1) for n in range(1, 200)) - 2 < Fr(1, 10**50),
      float(sum(Fr(int(sp.mobius(n)), 2**n - 1) for n in range(1, 200))))
N = 3000
S = sum(1 / (a * a * b * b) for a in range(1, N) for b in range(1, N // a + 1) if math.gcd(a, b) == 1)
print("coprime sum, truncated at ab<=3000:", S, "(limit 2.5)")

**Problems 1.13 to 1.18 (chance meets structure).**

In [ ]:
# 1.13: exact walk on a hexagon versus the formula
k, n = 6, 9
dist = np.zeros(k); dist[0] = 1
for _ in range(n):
    dist = 0.5 * (np.roll(dist, 1) + np.roll(dist, -1))
print(dist[0], (1 + 2 * 2**-n + 2 * (-0.5)**n + (-1)**n) / 6)
# 1.14: moments of fixed points for n = 5
n = 5
perms = list(itertools.permutations(range(n)))
mom = [Fr(sum(sum(s[i] == i for i in range(n))**k for s in perms), len(perms)) for k in range(1, 8)]
print(mom, [sp.bell(k) for k in range(1, 8)])
# 1.15: classes of 2x2 matrices over F_p
for p in [3, 5, 7]:
    c = Counter()
    for a, b, cc, d in itertools.product(range(p), repeat=4):
        t, D = (a + d) % p, (a * d - b * cc) % p
        r = [x for x in range(p) if (x * x - t * x + D) % p == 0]
        c['scalar' if (b, cc) == (0, 0) and a == d else ['irreducible', 'double', 'split'][len(r)]] += 1
    print(p, dict(c))
# 1.16: carries
b, nd, T = 10, 6, 100000
tot = 0
for _ in range(T):
    c = 0
    for i in range(nd):
        c = int(random.randrange(b) + random.randrange(b) + c >= b); tot += c
print(tot / T, nd / 2 - (1 - b**-nd) / (2 * (b - 1)))
# 1.17: Laplace's needle, a = 1, l = 0.7
a, l, T = 1.0, 0.7, 200000
X, Y, th = np.random.rand(T), np.random.rand(T), np.random.rand(T) * np.pi
X2, Y2 = X + l * np.cos(th), Y + l * np.sin(th)
nv = (np.floor(X2) != 0).astype(int); nh = (np.floor(Y2) != 0).astype(int)
print(np.mean(nv + nh > 0), (4 * a * l - l * l) / (np.pi * a * a), np.mean(nv + nh == 2), l * l / (np.pi * a * a))
# 1.18: E det^2 = n! exactly for n <= 4
for n in range(1, 5):
    tot = sum(int(round(np.linalg.det(np.array(e).reshape(n, n))))**2 for e in itertools.product([1, -1], repeat=n * n))
    print(n, Fr(tot, 2**(n * n)))

**Problems 1.19 to 1.22 (longer crossings).**

In [ ]:
def paley(p):
    chi = lambda a: 0 if a % p == 0 else (1 if pow(a, (p - 1) // 2, p) == 1 else -1)
    S = np.zeros((p + 1, p + 1), dtype=int)
    S[0, 1:] = 1; S[1:, 0] = -1
    S[1:, 1:] = [[chi(b - a) for b in range(p)] for a in range(p)]
    return np.eye(p + 1, dtype=int) + S
print(all((paley(p) @ paley(p).T == (p + 1) * np.eye(p + 1)).all() for p in [3, 7, 11, 19, 23, 31, 43, 47, 59]))
# Rudin-Shapiro and the flat-polynomial bound
P, Q = np.array([1.]), np.array([1.])
for _ in range(8):
    P, Q = np.concatenate([P, Q]), np.concatenate([P, -Q])
z = np.exp(1j * np.linspace(0, 2 * np.pi, 8192, endpoint=False))
vp, vq = np.abs(np.polyval(P[::-1], z)), np.abs(np.polyval(Q[::-1], z))
print(len(P), vp.max(), np.sqrt(2 * len(P)), np.abs(vp**2 + vq**2 - 2 * len(P)).max())
e = lambda m: sum(1 for i in range(m.bit_length()) if (m >> i) & 3 == 3)
print(all(P[m] == (-1)**e(m) for m in range(len(P))))
n = len(P) - 1
print("Problem 1.20 bound:", 2 * np.sqrt((n + 1) * np.log(5 * n * (n + 1))) + np.pi / 2)
# finite subgroup of order 12 in GL_2(Z)
A, B = np.array([[0, -1], [1, 1]]), np.array([[0, 1], [1, 0]])
G, frontier = {(1, 0, 0, 1)}, [np.eye(2, dtype=int)]
while frontier:
    new = []
    for M in frontier:
        for g in (A, B):
            N = M @ g
            if tuple(N.flatten()) not in G:
                G.add(tuple(N.flatten())); new.append(N)
    frontier = new
print(len(G), math.gcd(*[(p * p - 1) * (p * p - p) for p in sp.primerange(3, 60)]))

## Part B: laboratory

**L1.1 (★★) Sicherman dice with more faces.** Write a program that finds all pairs of $n$-faced dice (positive faces) with the
sum law of two ordinary $n$-faced dice, by distributing the cyclotomic factors $\Phi_d(x)$, $d\mid n$, $d>1$, of
$x\frac{x^n-1}{x-1}$ between the two dice (see the remark after Problem 1.1). Tabulate the number of pairs for $n\le30$.
For which $n$ is the ordinary pair the only one?

**L1.2 (★★) CS application: Hadamard codes and the Walsh--Hadamard transform.** The rows of a Hadamard matrix of order $n$
and their negatives form a binary code of $2n$ words in which two distinct words differ in at least $\frac n2$ positions.
(a) Build the code from the Paley matrix of order $12$ and check its minimum distance. (b) Implement the fast
Walsh--Hadamard transform for Sylvester matrices of order $2^k$ (cost $k2^k$) and use it to decode the first-order
Reed--Muller code: encode a random message, flip $15\%$ of the bits, and decode by taking the largest coefficient of the
transform. Measure the success rate for $k=5,\dots,10$. (The Mariner 9 probe used this code to send pictures of Mars in 1971.)

**L1.3 (★★) How flat can a Littlewood polynomial be?** For each $n\le20$, find by exhaustive search the minimum of $\|P\|$
over all sign choices $P=\sum_{k\le n}\pm z^k$, and plot $\min\|P\|/\sqrt{n+1}$. Compare with random signs and with
Rudin--Shapiro. (Evaluate $\|P\|$ on $2048$ points; why is that accurate enough? Use Problem 1.20's derivative bound.)

**L1.4 (★) Random determinants.** Sample $2000$ random $\pm1$ matrices of size $n=10,20,40,80$ and plot the histogram of
$\log|\det A|-\frac12\log(n-1)!$. Nguyen and Vu proved that, divided by $\sqrt{\frac12\log n}$, it tends to a standard normal
law. Do you see it? Compare also $\mathbb E[\det^2]$ with $n!$ (Problem 1.18) and explain why the empirical mean of
$\det^2$ is so unreliable.

## Solutions

### L1.1

In [ ]:
def cyclo_factors(n):
    return [sp.cyclotomic_poly(d, x) for d in sp.divisors(n) if d > 1]

def sicherman(n):
    fac = cyclo_factors(n)
    target = sp.Poly(sp.expand(sum(x**k for k in range(1, n + 1))**2), x)
    sols = set()
    for split in itertools.product(range(3), repeat=len(fac)):
        A = x
        for f, e in zip(fac, split):
            A *= f**e
        A = sp.Poly(sp.expand(A), x)
        if A.eval(1) != n:
            continue
        B = sp.Poly(sp.quo(target.as_expr(), A.as_expr()), x)
        if min(A.coeffs()) >= 0 and min(B.coeffs()) >= 0 and B.eval(0) == 0 and A.eval(0) == 0:
            fa = tuple(sorted(k for (k,), c in A.terms() for _ in range(int(c))))
            fb = tuple(sorted(k for (k,), c in B.terms() for _ in range(int(c))))
            sols.add(tuple(sorted([fa, fb])))
    return sols

counts = {n: len(sicherman(n)) for n in range(2, 31)}
print(counts)
print("only the ordinary pair:", [n for n, c in counts.items() if c == 1])

The ordinary pair is the only one exactly when $n$ is prime (then $x\frac{x^n-1}{x-1}=x\Phi_n$ has a single cyclotomic factor,
which must go to one die, and the value at $1$ forces it to be split $1+1$). For $n=4$ there is one extra pair
($1,2,2,3$ and $1,3,3,5$), for $n=6$ Sicherman's.

### L1.2

In [ ]:
H = paley(11)
code_words = np.vstack([H, -H])
dmin = min(np.sum(u != v) for u, v in itertools.combinations(code_words, 2))
print("Paley code: words", len(code_words), "length", H.shape[1], "min distance", dmin)

def fwht(a):
    a = a.astype(float).copy(); h = 1
    while h < len(a):
        for i in range(0, len(a), 2 * h):
            u, v = a[i:i + h].copy(), a[i + h:i + 2 * h].copy()
            a[i:i + h], a[i + h:i + 2 * h] = u + v, u - v
        h *= 2
    return a

def rm1_encode(msg, k):
    # message: k+1 bits (a0, a); codeword c_x = a0 + <a, x> mod 2 for x in F_2^k
    a0, a = msg[0], msg[1:]
    xs = np.array([[(i >> j) & 1 for j in range(k)] for i in range(2**k)])
    return (a0 + xs @ a) % 2

def rm1_decode(word, k):
    t = fwht(1 - 2 * word)                # +-1 signal, transform
    j = int(np.argmax(np.abs(t)))
    a = np.array([(j >> i) & 1 for i in range(k)])
    return np.concatenate([[0 if t[j] > 0 else 1], a])

for k in range(5, 11):
    ok = 0
    for _ in range(300):
        msg = np.random.randint(0, 2, k + 1)
        w = rm1_encode(msg, k)
        flips = np.random.rand(2**k) < 0.15
        ok += np.array_equal(rm1_decode((w + flips) % 2, k), msg)
    print(k, ok / 300)

The minimum distance of the Paley code is $6=\frac{12}2$, as predicted. The Reed--Muller decoder succeeds more and more
often as $k$ grows: with $15\%$ errors the expected number of flips, $0.15\cdot2^k$, stays well below the half-distance
$2^{k-2}$, and the fluctuations shrink relative to it.

### L1.3

In [ ]:
zz = np.exp(1j * np.linspace(0, 2 * np.pi, 2048, endpoint=False))
V = np.vander(zz, 21, increasing=True)
best = []
for n in range(1, 21):
    m = np.inf
    for signs in itertools.product([1, -1], repeat=n):    # first sign fixed to +1 by symmetry
        c = np.array((1,) + signs)
        m = min(m, np.abs(V[:, :n + 1] @ c).max())
    best.append(m / np.sqrt(n + 1))
rand = [np.mean([np.abs(V[:, :n + 1] @ np.random.choice([-1, 1], n + 1)).max() for _ in range(200)]) / np.sqrt(n + 1)
        for n in range(1, 21)]
plt.plot(range(1, 21), best, 'o-', label='best signs')
plt.plot(range(1, 21), rand, 's-', label='random signs (mean)')
plt.axhline(np.sqrt(2), ls='--', c='gray', label=r'Rudin--Shapiro bound $\sqrt{2}$')
plt.xlabel('n'); plt.ylabel(r'$\|P\|/\sqrt{n+1}$'); plt.legend(); plt.show()

On $2048$ points the grid spacing is $\frac{2\pi}{2048}$, and $|P'|\le\frac{n(n+1)}2\le210$, so the true maximum exceeds the
grid maximum by at most $\frac{\pi}{2048}\cdot210<0.33$, small against $\sqrt{n+1}$. The best ratio stays between $1.1$ and
$1.4$; random signs are worse and slowly growing, like $\sqrt{\log n}$.

### L1.4

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for axis, n in zip(ax, [10, 20, 40, 80]):
    sl = [np.linalg.slogdet(np.random.choice([-1., 1.], (n, n))) for _ in range(2000)]
    d = [ld for sg, ld in sl if sg != 0]           # drop the (rare) singular matrices
    d = (np.array(d) - 0.5 * math.lgamma(n)) / np.sqrt(0.5 * np.log(n))
    axis.hist(d, bins=40, density=True)
    t = np.linspace(-4, 4, 200); axis.plot(t, np.exp(-t * t / 2) / np.sqrt(2 * np.pi))
    axis.set_title(f"n={n}")
plt.show()
n = 10
dets = np.array([np.linalg.det(np.random.choice([-1., 1.], (n, n)))**2 for _ in range(20000)])
print(dets.mean() / math.factorial(n), "(should be near 1, but noisy)")

The histograms approach the Gaussian, slowly. The empirical mean of $\det^2$ is unreliable because $\det^2$ is heavy-tailed:
most of its expectation $n!$ comes from rare matrices with an unusually large determinant, which a sample of moderate size
rarely contains. The logarithm tames the tail, which is why one studies $\log|\det A|$.